# Bangkok Traffic — Data Feasibility และ Matching Audit

ฉบับร่าง ตามข้อตกลงผู้ใช้ล่าสุด ขอบเขตมีนาคม 2017–พฤษภาคม 2026 ยังไม่มี EDA หรือกราฟ ก่อนผู้ใช้ตรวจ audit

## Context & Methods
ใช้ `covid_period` ตามไฟล์; `Date/month/year` คือวันที่รายงาน; `survey_date` คือวันสำรวจ; ช่องว่างและ `-` ของจำนวนรถ 6 ประเภทเป็น 0 พร้อม log ไม่เปลี่ยน Excel ต้นฉบับ

### Key Assumptions
ชื่อปรับเฉพาะ Unicode/whitespace; ข้อมูลซ้ำและตัวเลขกำกวมคงไว้ใน staging และกันออกจาก candidate ชั่วคราว; เดือนรายงานไม่ใช่เดือนสำรวจ ข้อมูลปี 2026 เป็น YTD

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from traffic_audit import run_audit, normalize, survey_date, time_bounds
from write_audit_report import write_report
pd.set_option('display.max_rows', 12)
pd.set_option('display.max_columns', 10)
INPUT = Path(os.environ.get('TRAFFIC_INPUT', str(ROOT / 'data/raw/traffic_6e1b556f353c.xlsx')))
OUTPUT = ROOT / 'outputs/current_audit'

## Data
อ่านชีต traffic_data และรันกติกาเดิมทุกครั้ง ผลลัพธ์อ้างถึง snapshot และเลขแถว Excel ได้ เมื่อมีข้อมูลใหม่ให้ตั้ง TRAFFIC_INPUT เป็นพาธไฟล์นั้นแล้ว Run All

In [2]:
summary = run_audit(INPUT, OUTPUT)
write_report()
print(f"Rows: {summary['rows']:,}; candidate rows: {summary['candidate_rows']:,}")
print('Source SHA-256:', summary['source_sha256'])
print('Period counts:', summary['period_counts'])

Rows: 17,005; candidate rows: 16,829
Source SHA-256: 6e1b556f353c66fec5c8a664f3e1e5b5e52e5a756f75c2ae28562c5c0fa6118b
Period counts: {'หลังโควิด': 8619, 'ก่อนโควิด': 4917, 'โควิด': 3469}


## Results
### Coverage ตามปีรายงาน
ปี 2017 เริ่มมีนาคม และปี 2026 สิ้นสุดพฤษภาคมตามขอบเขต ไม่ถือว่าเดือนนอกขอบเขตหาย

In [3]:
display(pd.DataFrame(summary['year_coverage']))

,report_year,rows,months_present,months_missing_within_scope,locations,assumed_zero_rows
0,2017,1422,"3,4,5,6,7,8,9,10,11,12",,476,0
1,2018,1671,"1,2,3,4,5,6,7,8,9,10,11,12",,531,0
2,2019,1824,"1,2,3,4,5,6,7,8,9,10,11,12",,611,0
3,2020,1333,"1,2,3,4,5,7,8,10,11,12","6,9",420,560
4,2021,2136,"1,2,3,4,5,6,7,8,9,10,11,12",,677,549
5,2022,2048,"1,2,3,4,5,6,7,8,9,10,11,12",,671,400
6,2023,1923,"1,2,3,4,5,6,7,8,9,10,11,12",,627,0
7,2024,1875,"1,2,3,4,5,6,7,8,9,10,11,12",,618,0
8,2025,2062,"1,2,3,4,5,6,7,8,9,10,11,12",,657,0
9,2026,711,"1,2,3,4,5",,233,0


### ผลการแทน 0 และค่าที่ต้องตรวจ
จำนวนเซลล์ไม่เท่ากับจำนวนแถว ค่า 0 ที่แทนเป็นสมมติฐานที่ผู้ใช้กำหนด

In [4]:
display(pd.DataFrame(summary['numeric_profile']))
print('Rows with assumed zeros:', summary['zero_assumed_rows'])
display(pd.read_csv(OUTPUT / 'numeric_review.csv'))

,field,blank,dash,zero_assumed,unresolved,negative,fractional
0,passenger_car,0,31,31,0,0,1
1,pickup_van,0,33,33,0,0,3
2,large_bus,0,428,428,0,0,2
3,small_bus,0,1458,1458,0,0,0
4,truck,0,118,118,0,0,0
5,tuk_tuk,0,220,220,0,0,0


Rows with assumed zeros: 1509


,source_excel_row,passenger_car,pickup_van,large_bus,small_bus,truck,tuk_tuk
0,11615,1153.96,595.0,1.0,0,14,2
1,11977,9972.00,3049.7,161.0,0,291,58
2,12043,3649.00,1363.3,113.0,0,90,85
3,12249,2753.00,868.4,38.0,1,30,35
4,13759,33383.00,12410.0,568.7,64,918,704
5,15670,33383.00,12410.0,568.7,64,918,704


### Matching Audit
ทุกกลุ่มมี covid_period ครบสามช่วง แต่ไม่รับรองว่ามีทุกปี `groups` คือชุดคีย์; `locations` คือคู่ทางแยก+ถนน; `rows` คือแถวต้นทาง

In [5]:
display(pd.DataFrame(summary['matching']))

,design,groups,locations,intersections,rows,before_rows,during_rows,after_rows
0,raw_location,357,357,177,5072,1587,1295,2190
1,candidate_location,357,357,177,5047,1587,1289,2171
2,candidate_location_time,623,352,175,2612,902,733,977
3,candidate_report_month_time,15,11,6,47,17,15,15
4,candidate_report_quarter_time,70,52,29,226,81,70,75
5,candidate_survey_month_time,15,11,6,47,17,15,15


### Coverage ของชุดไตรมาสรายงาน
ตรวจปีจริงที่เหลือ ห้ามใช้จำนวนแถวทั้งชุดแทน sample size ของปี 2026

In [6]:
quarter = pd.read_csv(OUTPUT / 'candidate_report_quarter_time_rows.csv')
display(quarter.groupby(['report_year', 'covid_period']).size().reset_index(name='rows'))

,report_year,covid_period,rows
0,2017,ก่อนโควิด,27
1,2018,ก่อนโควิด,25
2,2019,ก่อนโควิด,29
3,2020,โควิด,31
4,2021,โควิด,39
5,2022,หลังโควิด,30
6,2023,หลังโควิด,14
7,2024,หลังโควิด,8
8,2025,หลังโควิด,17
9,2026,หลังโควิด,6


## Checks
ตรวจ parser เวลา/วันที่ และคำนวณ intersection ของเซตสถานที่อย่างอิสระจาก groupby เพื่อยืนยันผล matching

In [7]:
assert survey_date('(18 เม.ย. 65)') == pd.Timestamp('2022-04-18')
assert time_bounds('เร่งด่วนเช้า (07:00-09:00)') == time_bounds('เร่งด่วนเช้า (7.00 - 9.00 น.)')
assert time_bounds('นอกเร่งด่วน (9.00 - 17.00 น.)')[2] == 8
staging = pd.read_csv(ROOT / 'data/interim/traffic_audited.csv')
sets = [set(zip(g.intersection_key, g.road_key)) for _, g in staging.groupby('covid_period')]
assert len(set.intersection(*sets)) == summary['matching'][0]['locations']
assert len(staging) == summary['rows']
assert staging['passenger_car_zero_assumed'].sum() == summary['numeric_profile'][0]['zero_assumed']
print('Independent matching, source-preservation, and parser checks passed.')

Independent matching, source-preservation, and parser checks passed.


C:\Users\AphatsaraKhangkhet(C\AppData\Local\Temp\ipykernel_34708\3505366427.py:4: DtypeWarning: Columns (0: passenger_car, 1: pickup_van, 2: large_bus, 3: small_bus, 4: truck, 5: tuk_tuk, 6: Unnamed: 16, 7: Unnamed: 17) have mixed types. Specify dtype option on import or set low_memory=False.
  staging = pd.read_csv(ROOT / 'data/interim/traffic_audited.csv')


## Takeaways
เริ่มทำ Mini Project แบบมีเงื่อนไขได้ เลือก matching ให้ตรงกับคำถามก่อน EDA; กลุ่มรายเดือนตรงกันมีขนาดเล็ก ชุดไตรมาสรายงานเป็นทางเลือกแต่ไม่ควบคุมฤดูกาลสำรวจโดยสมบูรณ์ ตรวจข้อมูลซ้ำ/ตัวเลขกำกวมและชื่อสถานที่ก่อนยืนยันตัวอย่าง อ่านรายงาน outputs/current_audit/Data_Feasibility_Matching_Audit.md ที่สร้างจากผลรันนี้

AI ช่วยสร้างโค้ดและเอกสาร สมาชิกต้องทวนข้อมูลต้นทางและตัดสินขอบเขต ยังไม่ได้สรุปแนวโน้ม การฟื้นตัว หรือสาเหตุจาก COVID